**BANK CUSTOMER CHURN PREDICTION**
###### - ANANYA PATANKAR


---


In [1]:
#import required libraries
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
from sklearn.compose import ColumnTransformer
from imblearn.over_sampling import SMOTE
import joblib

In [2]:
#load the dataset
df = pd.read_csv('bank churn dataset.csv')

**DATA CLEANING :**

In [3]:
#drop irrelevant columns
df.drop(columns= ['RowNumber','CustomerId','Surname'],inplace=True)

Drop identifiers like RowNumber, CustomerId, and Surname since they are irrelevant in predicting customer churn. Also since RowNumber and CustomerId are numeric columns, they may be treated as numeric features by the model, creating noise. Surname is a categorical column with no predictive value.

In [4]:
#rename some columns for better readability
df.rename(columns = {'Geography':'Country','HasCrCard':'HasCreditCard','IsActiveMember':'IsActive', 'EstimatedSalary':'Salary'}, inplace = True)

Columns like Geography, HasCrCard, IsActiveMember, and EstimatedSalary are renamed to Country, HasCreditCard, IsActive, and Salary respectively for better readability and clarity regarding column data.

**FEATURE ENGINEERING :**

In [5]:
#churn rate by number of products
df.groupby('NumOfProducts')['Exited'].mean()

,Exited
NumOfProducts,
1,0.277144
2,0.075817
3,0.827068
4,1.000000


In [6]:
#create threshold flag to identify safe vs at-risk customers
df['ProductsRisk'] = (df['NumOfProducts']!=2).astype(int)

Churn rates by number of products used show a nonlinear pattern. Customer having exactly two products have a significantly lower churn rate (7.58%) and hence are more stable than those having 1,3 or 4 products. We add a threshold flag to differentiate between the safer customers (number of products =  2) and at-risk customers (number of products = 1,3,4).

In [7]:
#churn rate comparison of zero and non-zero balance customers
print("Churn rate- Zero balance:", df[df['Balance']==0]['Exited'].mean())
print("Churn rate- Non-zero balance:", df[df['Balance']>0]['Exited'].mean())

print()

#country wise breakup of zero balance customers
print(df[df['Balance']==0]['Country'].value_counts())

Churn rate- Zero balance: 0.13823610727121924
Churn rate- Non-zero balance: 0.2407958640137866

Country
France    2418
Spain     1199
Name: count, dtype: int64


In [8]:
#create binary flag to identify zero balance customers
df['ZeroBalance'] = (df['Balance']==0).astype(int)

Zero balance customers churn less (13.82%) than those with non-zero balance (24.08%). This paradox suggests that these could be secondary or dormant accounts, with the customers being indifferent rather than unhappy with the bank.
Additionally zero balance customers are exclusively from France and Spain, not Germany, i.e. Germany, the country with highest churn (32%) has no customers with zero balance.
Hence we create a binary flag to separate and understand the distinct behavioural pattern of zero balance customers.

In [9]:
#churn rate by age group
df['AgeGroup'] = pd.cut(df['Age'], bins=[16,40,60,100], labels=['Young','Middle','Old'])
df.groupby(['AgeGroup','IsActive'], observed = True)['Exited'].mean()

AgeGroup  IsActive
Young     0           0.134772
          1           0.079168
Middle    0           0.510883
          1           0.281672
Old       0           0.820225
          1           0.112000
Name: Exited, dtype: float64

In [10]:
#create interaction feature to get combined effect of age and activity status
df['AgeInactive'] = df['Age']*(1-df['IsActive'])

Older customers and inactive customers showed higher churn rates individually in EDA. On studying the combined effects of age and activity status, it was found senior inactive customers had a significantly higher churn rate of 82% compared to senior active customers (11.2%). Thus older inactive customers at high risk of churn.
Hence we create an interaction feature AgeInactive= Age * (1-IsActive) which activates for only inactive members, showing their age as risk signal, otherwise shows 0 for active members.

In [11]:
#create ratio to normalize tenure by age
df['TenureAgeRatio'] = df['Tenure']/df['Age']

Tenure has almost zero correlation with churn rate (-0.014). However tenure has a different meaning at different ages. A customer with 5 years tenure at 30 (joined at 25) is more loyal compared to a customer with a 5 years tenure at 60 (joined at 55). By dividing tenure by age, we normalize it relative to age, making it a better loyalty signal.

In [12]:
#create ratio to normalize balance by salary
df['BalanceSalaryRatio'] = df['Balance']/df['Salary']

Balance cannot be used to understand a customer's financial status on its own as it is relative to salary. A customer with a balance of 100,000 with salary 150,000 is different from a customer having 100,000 balance with 50,000 salary. Hence we normalize balance by dividing it by salary to get a better measure of customer engagement.

Verify changes :

In [13]:
df.shape

(10000, 17)

In [14]:
df.head()

,CreditScore,Country,Gender,Age,Tenure,Balance,NumOfProducts,HasCreditCard,IsActive,Salary,Exited,ProductsRisk,ZeroBalance,AgeGroup,AgeInactive,TenureAgeRatio,BalanceSalaryRatio
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1,1,1,Middle,0,0.047619,0.000000
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0,1,0,Middle,0,0.024390,0.744677
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1,1,0,Middle,42,0.190476,1.401375
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0,0,1,Young,39,0.025641,0.000000
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0,1,0,Middle,0,0.046512,1.587055


In [15]:
df.isna().sum()

,0
CreditScore,0
Country,0
Gender,0
Age,0
Tenure,0
Balance,0
NumOfProducts,0
HasCreditCard,0
IsActive,0
Salary,0


The newly added features are reflecting correctly on the dataframe and no null values are found. The dataset is ready for the next preprocessing steps.

**TRAIN/TEST SPLIT:**

In [16]:
#separate data into features and target
X = df.drop(columns="Exited")
y = df["Exited"]

In [17]:
#split data into train and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y)

The data is split into training and test sets in a 80-20 ratio, with 8000 rows in training and 2000 in test. The original class distribution of ~80% retained customers and ~20% churned customers is maintained using stratification.

Verify split :

In [18]:
#check shape of each set
print('X_train:', X_train.shape)
print('X_test:', X_test.shape)
print('y_train:', y_train.shape)
print('y_test:', y_test.shape)

X_train: (8000, 16)
X_test: (2000, 16)
y_train: (8000,)
y_test: (2000,)


In [19]:
#check if class imbalance is preserved
print('y_train distribution:')
print(y_train.value_counts(normalize=True))
print()
print('y_test distribution:')
print(y_test.value_counts(normalize=True))

y_train distribution:
Exited
0    0.79625
1    0.20375
Name: proportion, dtype: float64

y_test distribution:
Exited
0    0.7965
1    0.2035
Name: proportion, dtype: float64


The train/test split is completed successfully with train(80%) and test(20%). The class imbalance is also maintained.

**ENCODING CATEGORICAL DATA:**

In [20]:
#drop AgeGroup column
X_train.drop(columns=['AgeGroup'], inplace=True)
X_test.drop(columns=['AgeGroup'], inplace=True)

The insights from the AgeGroup column are captured in the AgeInactive column. Thus it is a redundant column now and can be dropped.

In [21]:
#one-hot encode Country and Gender
X_train = pd.get_dummies(X_train, columns=['Country', 'Gender'], drop_first=True)
X_test  = pd.get_dummies(X_test,  columns=['Country', 'Gender'], drop_first=True)

One-Hot Encoding is applied to columns Country and Gender as they are categorical in nature. 'drop_first = True' is used to drop one category from each column to avoid multicollinearity.

In [22]:
#align columns
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

X_test columns are aligned with X_train to ensure that both have identical columns after encoding.

Verify encoding :

In [23]:
print(X_train.shape)
print(X_test.shape)
print(X_train.columns.tolist())

(8000, 16)
(2000, 16)
['CreditScore', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCreditCard', 'IsActive', 'Salary', 'ProductsRisk', 'ZeroBalance', 'AgeInactive', 'TenureAgeRatio', 'BalanceSalaryRatio', 'Country_Germany', 'Country_Spain', 'Gender_Male']


The encoding of categorical data is completed successfully. Country and Gender columns are replaced by Country_Germany, Country_Spain, and Gender_Male. Both X_train and X_test have similar shapes.

**FEATURE SCALING :**

In [24]:
#scale features using different scalers based on the distribution of each column
preprocessor = ColumnTransformer(transformers=[
    ('std', StandardScaler(), ['CreditScore','Age','Tenure']),
    ('minmax', MinMaxScaler(), ['NumOfProducts']),
    ('robust', RobustScaler(), ['Balance','Salary','AgeInactive','BalanceSalaryRatio']) ],
    remainder='passthrough')

The features in the dataset have different ranges (eg. Credit Score ranges from 350 to 850, balance from 0 to 250898.09, etc). Scaling normalizes these features to prevent one feature from dominating the model.

In [25]:
#fit on train set only
X_train_scaled = preprocessor.fit_transform(X_train)
X_test_scaled  = preprocessor.transform(X_test)

The scaler is fit on train set only to avoid data leakage into the test set.

Verify scaling :

In [26]:
print(preprocessor.get_feature_names_out())
pd.DataFrame(X_train_scaled).head()

['std__CreditScore' 'std__Age' 'std__Tenure' 'minmax__NumOfProducts'
 'robust__Balance' 'robust__Salary' 'robust__AgeInactive'
 'robust__BalanceSalaryRatio' 'remainder__HasCreditCard'
 'remainder__IsActive' 'remainder__ProductsRisk' 'remainder__ZeroBalance'
 'remainder__TenureAgeRatio' 'remainder__Country_Germany'
 'remainder__Country_Spain' 'remainder__Gender_Male']


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15
0,1.058568,1.715086,0.684723,0.0,-0.760422,0.613559,1.540541,-0.496009,1,0,1,1,0.122807,False,False,True
1,0.913626,-0.659935,-0.696202,0.0,0.039748,-0.362501,0.864865,0.559996,1,0,1,0,0.09375,True,False,True
2,1.079274,-0.184931,-1.731895,0.333333,0.131706,0.183463,0.0,0.14566,1,1,0,0,0.0,True,False,False
3,-0.929207,-0.184931,-0.005739,0.333333,-0.760422,-0.167154,1.0,-0.496009,1,0,0,1,0.135135,False,False,True
4,0.427035,0.955079,0.339492,0.333333,0.105657,0.082036,0.0,0.18444,0,1,0,0,0.122449,True,False,True


Feature scaling is completed successfully. Continuous features are normalized, discrete features are within the range of 0 to 1, while binary features remain unchanged.

**RESAMPLING :**

In [27]:
#apply SMOTE to training data
sm = SMOTE(random_state=42)
X_train_final, y_train_final = sm.fit_resample(X_train_scaled, y_train)

SMOTE is used to correct the class imbalance observed during EDA. Synthetic samples are generated for the minority class (churned customers) to match the majority class (retained customers).

Verify resampling :

In [28]:
print((y_train_final).value_counts())
print((y_train_final).value_counts(normalize=True))

Exited
1    6370
0    6370
Name: count, dtype: int64
Exited
1    0.5
0    0.5
Name: proportion, dtype: float64


Resampling is completed successfully. Both classes are now equally distributed with 6370 samples, each representing 50% of the training data.

**SAVING PROCESSED DATA :**

In [29]:
#save scaled and resampled training data
pd.DataFrame(X_train_final).to_csv('X_train_final.csv', index=False)
pd.DataFrame(X_test_scaled).to_csv('X_test_final.csv', index=False)

In [30]:
#save target variables
y_train_final.to_csv('y_train_final.csv', index=False)
y_test.to_csv('y_test_final.csv', index=False)

In [31]:
#save preprocessor
joblib.dump(preprocessor, 'preprocessor.pkl')

['preprocessor.pkl']

The data preprocessing stage is complete. The processed datasets are saved for use in the modelling stage.